In [32]:
import sys
sys.path.append('../')

import MESAGen as mg
import mesa_reader as mr
import time
import InlistHelper as ih
from bisect import bisect_right
import numpy as np
import CEconstants as const

In [33]:
filename = '../StarModels/Fragos/12MsunV1.mod'

In [34]:
def getPointIndex(R, x):
    R = R[::-1]
    realIndex = bisect_right(R, x)
    modIndex = len(R)-realIndex
    return modIndex

In [35]:
def getAccretionZones(R, a, xhigh, xlow, limit=1):
    highIndex = getPointIndex(R, xhigh)
    lowIndex  = getPointIndex(R, xlow)
    if (highIndex < limit):
         return np.arange(limit, lowIndex+1)
    return np.arange(highIndex, lowIndex+1)

In [36]:
def getIsotopes(modData, zones):
    presentIsotopes = [iso for iso in const.commonIsotopes if iso in modData]
    isotopeData = {iso: modData[iso][zones] for iso in presentIsotopes}

    # Use for debugging, make sure we are depositing in the correct radial zones...
    #isotopeData['lnR']   = modData['lnR'][zones]
    #isotopeData['Index'] = zones
    
    return isotopeData

In [49]:
# TEST Grounds.
modData = mg.readModfile(filename)
lnR      = ih.changeDEModfile(modData['lnR'], 'D', 'E')
lnT      = ih.changeDEModfile(modData['lnT'], 'D', 'E')
dq       = ih.changeDEModfile(modData['dq'], 'D', 'E')
R        = np.exp(lnR)
Rupper = 4.21033997e+13
Rlower = 1.62164062e+07
zones = getAccretionZones(R, 4.21033997e+10, Rupper, Rlower)
test = getIsotopes(modData, zones)


isotopes = list(test.keys())
species = len(isotopes)
xaLst   = list(test.values())
#xval = list(test.values())
#test3 = ih.changeDEModfile(xval[0], 'D', 'E')

#for a in range(0, len(isotopes)):
#    print(isotopes[a], test[isotopes[a]][0])
print(len(zones))
print(len(xaLst[0]))

for i, zone in enumerate(zones):
    #print('i: ', i)
    xa = np.array([xaLst[j][i] for j in range(species)])
    xa = ih.changeDEModfile(xa, 'D', 'E')
    #print(xa)


1660
1660


In [21]:
def addInternalEnergy(pyMesaObject, modData, zones, Ezones, isotopeData, totalStarMass, args):
    dqLst   = args[0]
    tempLst = args[1]
    rhoLst  = args[2]
    istopes = list(isotopeData.keys())
    xaLst   = list(isotopeData.values())
    species = len(isotopes)
    chem_ids = np.zeros(species)

    #print('Isotopes List: ', isotopes)
    #print('Zones: ', zones)
    #print('xalst: ', xalst)
    #print('ENergy to deposit: ', Ezones)
    
    for i, name in enumerate(isotopes):
        chem_ids[i] = context.chem_lib.chem_get_iso_id(name).result

    net_iso = chem_ids
    
    for i, zone in enumerate(zones):
        dq     = dqLst[zone]    # Mass Fraction
        T      = tempLst[zone]  # Tempertaure
        rho    = rhoLst[zone]   # Density
        Einput = Ezones[i]            # Energy to deposit
        #xa     = xaLst[i]           # Isotope Mass Fraction
        #xa     = np.array(xa)
        xa = np.array([xaLst[j][i] for j in range(species)])
        xa = ih.changeDEModfile(xa, 'D', 'E')

        #print('==============================================================')
        # Calls Mesa EOS to get Internal Energy
        eos_result = getEosResult_IE(context, rho, T, species, chem_ids, net_iso, xa)
        res        = eos_result.args["res"]
        i_lnE      = context.eos_def.i_lne - 1  # index for Internal Energy
        IE         = np.exp(res[i_lnE])
        #print('Internal Energy:   {} erg/g'.format(IE))
        
        # Adding internal energy
        newIE = IE + (Einput/(dq*Msun*starMass))
        #newIE = IE # Test that we get the same internal energy
        #print('New Internal Energy: {} erg/g'.format(newIE))
    
        # Calls Mesa EOS to get new Temperature.... Need to write as nautural log for mod file
        T = getEosResult_Temperature(context, rho, T, species, chem_ids, net_iso, xa, newIE, i_lnE)
        #print('New addded Ln T: ', np.log(T))
        
        #modData['lnT'][zones[a]] = str(np.log(T))
        modData['lnT'][zones[a]] = f"{np.log(T):.16E}"
    return modData

In [ ]:
# NOTE: Aagin the modfile way reads the arrays from larger radius to inner
#       It doesnt matter for gettignt the distribution, but make sure that
#       every other place where calculations are made its consistent.
def getEnergyDistribution(R, zones, dq, starM, Einput, kernel):
    firstZone = zones[0]
    lastZone  = zones[-1]
    accRadius = R[firstZone-1:lastZone]
    dq        = dq[firstZone - 1: lastZone] * starM
    totalM    = sum(dq)
    fraction  = np.linspace(0, 1, len(dq))  # Create an array of fractions from 0 to 1

    if (kernel == 'Triangle'):
        # Triangular distribution
        c = 0.5  # Center of the triangular distribution (adjust as needed)
        weights = triang.pdf(fraction, c)
    elif(kernel == 'Parabolic'):
        # Parabolic distribution
        weights = 1 - (2 * fraction - 1)**2
    elif(kernel == 'Cosine'):
        mu = 0.5  # Center of the cosine distribution (adjust as needed)
        weights = 0.5 * (1 + np.cos((fraction - mu) * np.pi))
    elif(kernel == 'Quartic'):
        mu = 0.5  # Center of the quartic distribution (adjust as needed)
        weights = (1 - (fraction - mu)**2)**2
    elif(kernel == 'Triweight'):
        weights = (35 / 32) * (1 - (fraction**2))**3
    elif(kernel == 'Gauss'):
        mu = 0.5  # Mean of the Gaussian distribution (adjust as needed)
        sigma = 0.1  # Standard deviation of the Gaussian distribution (adjust as needed)
        weights = norm.pdf(fraction, loc=mu, scale=sigma)
    elif(kernel == 'Uniform'):
        weights = np.ones_like(fraction)
    elif(kernel == 'Os'):
        sigma = 0.1
        gaussian_center = np.exp(-(fraction - 0.5)**2 / (2 * sigma**2))
        weights = gaussian_center * (0.5 + 0.5 * np.sin(2 * np.pi * 50.0 * fraction))
        
    Ezones = Einput * (weights / sum(weights))
    return Ezones, AccR

In [53]:
def updateRunMesa(pyMesaObject, simParameters, runParameters, donorStar, modData):
    
    RupperDrag = simParameters.RupperDrag
    RlowerDrag = simParameters.RlowerDrag
    a          = simParameters.a
    Edrag      = simParameters.Edrag
    Ejet       = simParameters.Ejet
    dragKernel = runParmaeters.dragKernel
    jetKernel  = runParameters.jetKernel
    totalStarMass = donorStar.Mass
    dt        = runParameters.dtType

    # 1.Get log arrays from zone and change format to edit
    lnR      = ih.changeDEModfile(modData['lnR'], 'D', 'E')
    dq       = ih.changeDEModfile(data['dq'], 'D', 'E')
    lnT      = ih.changeDEModfile(data['lnT'], 'D', 'E')
    lnd      = ih.changeDEModfile(data['lnd'], 'D', 'E')
    starAge  = float(replaceChar(data['star_age'], 'D', 'E'))
    
    R        = np.exp(lnR)
    dq       = dq
    T        = np.exp(lnT)
    rho      = np.exp(lnd)
    args = [dq, T, rho]
    
    # 2. Get the zones to modify form the modfile based on the ranges
    #    a) This step is importnat to onyl call modfile and create data once since
    #       this takes up a lot time if have fucntions calling modfile to get data.
    zones = getAccretionZones(R, a, RupperDrag, RlowerDrag)

    # Get energy distribution.
    Ezones, AccRegion = getEnergyDistribution(R, zones, dq, totalStarMass,
                                             Edrag, dragKernel)

    # 3. Get the isotopes header defined, need to be done each iteartion since istopes can change
    isotopeData = getIsotopes(modData)

    # 4. Deposit the energy needed modifying the MESA Mod Data
    finalModData = addInternalEnergy(pyMesaObject, modData, zones, Ezones, isotopeData, 
                     totalStarMass, args)
    
    # TODO: Check the array lenghts of each
    
    # 5. Now Write the Modfied Mod data back into MOD file
    ms.writeModfile(finalModData, runParameters.modFile)  # Writing modfile with energy deposited...

    # 6. Run Mesa.
    ih.addAgeInlist(runParameters.inlistFile, dt)  # Increase inlist max_age...
    
    mesaStartTime = time.time()
    mg.RunMesa(runParameters.pathMesa, runParameters.numThreads)
    mesaFinalTime = time.time() - mesaStartTime
    print("Mesa Called...            {} [secs]".format(mesaFinalTime), flush = True)
    
    # At this point MESA has a new modfile present.
    # Maybe just return final mod data after reading once, and 
    #   set it up outside for the next iteration
    # since int setp 5, we always write it out prep for the next iteration
    # overwriting previous.
    finalModData = ms.readModfile(runParameters.finalMod)
    nextStarAge  = float(replaceChar(finalModData['star_age'], 'D', 'E'))
    #ms.writeModfile(newModFile, runParameters.modFile) #

    if (nextStarAge = starAge):
        print("MESA: Star did not evolve, check MesaOutPut.txt")
        simParameters.mesaFlag = False
        return finalModData

    return finalModData

In [55]:
test.keys()

dict_keys(['version_number', 'M/Msun', 'model_number', 'star_age', 'initial_z', 'n_shells', 'net_name', 'species', 'Teff', 'power_nuc_burn', 'power_h_burn', 'power_he_burn', 'power_z_burn', 'power_photo', 'total_energy', 'cumulative_energy_error', 'lnd', 'lnT', 'lnR', 'L', 'dq', 'mlt_vc', 'h1', 'he3', 'he4', 'c12', 'n14', 'o16', 'ne20', 'mg24', 'si28', 'headerVariables', 'Index', 'startComments', 'lastComments'])

In [90]:
wallclock = 0.0
for a in range(0, 8):
    startime = time.time()
    test = mg.readModfile('../StarModels/Fragos/12MsunV1.mod')
    wallclock += time.time()-startime
print('Total time: ', wallclock)

Total time:  0.29053425788879395


In [51]:
wallclock = 0.0

for a in range(0, 2):
    startime = time.time()
    test = TESTreadModfile('../StarModels/Fragos/12MsunV1.mod')
    wallclock += time.time()-startime
print('Total time: ', wallclock)

IndexError: index 0 is out of bounds for axis 1 with size 0

In [41]:
wallclock = 0.0

for a in range(0, 100):
    startime = time.time()
    test = mr.MesaData('../StarModels/Fragos/12MsunV1Profile.data')
    wallclock += time.time()-startime
    
print('Total time: ', wallclock)


KeyboardInterrupt

